<a href="https://colab.research.google.com/github/abubakarsaleem18/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarsaleem18/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My Lane as an ML Task

My lane (Refresh/Content Opportunity Scoring) is a **scoring / ranking task**,
not a strict classification task. Rather than sorting pages into hard
categories, the goal is to produce a continuous priority score per page, so
reviewers can sort pages from "most worth checking" to "least worth
checking." Under the hood this can be built on top of a classification
model (e.g. predicting probability of "needs review") — but the *output*
that matters is the ranked order, not a hard yes/no label.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or Proxy

I don't have a ground-truth "needs review" label, so I need a proxy target.
I'm defining a page as a positive example (`needs_review = 1`) if it has:
- `impressions_90d >= 500` (still visible enough to matter), AND
- `trend_direction == "down"` (currently declining)

All other pages are `needs_review = 0`.

This is a proxy, not ground truth — a page could be declining for reasons
unrelated to content quality (seasonality, a competitor change), so this
target captures "worth investigating," not "definitely broken."

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric

I'll use **Precision@50**: of the top 50 pages my model ranks highest, what
fraction are genuinely `needs_review = 1`? This matches how the output will
actually be used — a reviewer works through a fixed-size queue, not the
entire dataset, so precision at the top of the ranking matters more than
overall accuracy across all pages.

In [16]:
!git clone https://github.com/abubakarsaleem18/flyrank-internship-ml.git
%cd flyrank-internship-ml

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 132 (delta 46), reused 100 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.85 MiB | 11.49 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. Unit of Analysis

Each row in the dataframe above is **one content page**, identified by
`content_id`. The last column, `needs_review`, is my proxy target — the
column a model would eventually learn to predict.

In [18]:
import os
print(os.getcwd())
print(os.listdir())

/content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml
['LICENSE', 'AGENTS.md', 'CLAUDE.md', 'data', 'outputs', 'skills', 'work', '.git', 'SETUP.md', '.github', '.gitignore', 'requirements.txt', 'README.md', 'docs', 'notebooks', 'submission', 'scripts', 'DATA_USE.md', 'GUIDE.md']


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define the proxy target
df["needs_review"] = (
    (df["impressions_90d"] >= 500) & (df["trend_direction"] == "down")
).astype(int)

# Show the unit of analysis: one row = one content page
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"needs_review distribution:\n{df['needs_review'].value_counts(normalize=True)}")

df[["content_id", "impressions_90d", "trend_direction", "word_count", "needs_review"]].head(10)

Shape: 30000 rows, 45 columns
needs_review distribution:
needs_review
0    0.667967
1    0.332033
Name: proportion, dtype: float64


,content_id,impressions_90d,trend_direction,word_count,needs_review
0,content_304f48230142,3803,down,3221.0,1
1,content_a1fb4e703a9e,15320,down,2481.0,1
2,content_9aa793d4d895,12581,down,3515.0,1
3,content_331d6c4de07b,11751,stable,NaN,0
4,content_d99b7a2d90ca,19140,down,2803.0,1
5,content_d4084a4bc775,3970,down,3080.0,1
6,content_9a34b442b552,20,down,3059.0,0
7,content_a63219c6e95a,1724,stable,NaN,0
8,content_5e6c160719bc,32574,down,3807.0,1
9,content_c27558df2b0c,1240,down,NaN,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule Here

I already defined `needs_review` using a simple fixed rule (impressions +
trend direction) — and that rule alone is exactly the kind of baseline the
starter pipeline shows underperforming (Precision@50 of 0.240). A fixed
rule treats every signal independently and with the same weight for every
page. An ML model, by contrast, can learn *interactions* — for example, a
declining page with high historical backlinks might recover on its own,
while a declining page with thin content and no backlinks probably won't.
Those conditional patterns are hard to hand-write as an if/else rule, but a
model can learn them directly from data.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 6. Self-Check

- ✅ Named the task type: scoring/ranking.
- ✅ Defined the proxy target (`needs_review`) and explained why it's a proxy, not ground truth.
- ✅ Named the success metric: Precision@50.
- ✅ Showed the unit of analysis as a real dataframe (one row = one page).
- ✅ Explained why ML beats a fixed rule: it can capture interactions between signals.
- ✅ Tied the output to a real action: reviewers use the ranked list to prioritize which pages to check.